# Experiment 1 — BGE Directional Fine-Tuning

Main question: can triplet fine-tuning improve PRO-vs-CON leakage direction discrimination when training only on sentences already known to contain ballot leakage?




## Setup

Install dependencies, authenticate Colab, and open the source spreadsheet.


In [5]:
# SETUP — Install the packages required by this independent Colab notebook.
# The version-free installs preserve the historical notebook behavior while allowing a fresh runtime to resolve compatible packages.
!pip install -q sentence-transformers datasets accelerate gspread pandas scikit-learn


In [6]:
# SETUP 2 — Authenticate Colab and connect to Google Sheets.
# The authenticated gspread client is used only to read the frozen dataset tabs in this notebook.
from google.colab import auth
auth.authenticate_user()

import google.auth
import gspread
import pandas as pd
import numpy as np

creds, _ = google.auth.default()
gc = gspread.authorize(creds)

# Import display explicitly so table previews also work outside Colab's implicit namespace.
from IPython.display import display


In [7]:
# SETUP 3 — Open the ballot-leakage spreadsheet by URL.
# Listing worksheet titles confirms that TRAIN, VALIDATION, TEST, and summary tabs are available.
SHEET_URL = "https://docs.google.com/spreadsheets/d/1HnVdDmYMZnnEDZANBYWM3RJ0w7WEpXQGZrCxWkyoqMQ/edit"

spreadsheet = gc.open_by_url(SHEET_URL)

print([ws.title for ws in spreadsheet.worksheets()])


['Candidate Dataset', 'Source Ballots', 'Read Me', 'TRAIN', 'VALIDATION', 'TEST', 'SPLIT SUMMARY', 'BGE BASELINE VALIDATION', 'BGE BASELINE METRICS']


## Load Dataset

Load and normalize the frozen TRAIN and VALIDATION tabs without changing annotations.


In [8]:
# Cell 37 — Load TRAIN
# STEP 1 — Read the frozen training split and standardize its annotation fields.
# sentence is the model input; judge_id and ballot_id remain for traceability and overlap checks.
train_ws = spreadsheet.worksheet("TRAIN")
train_df = pd.DataFrame(
    train_ws.get_all_records()
)

# Convert strength labels to numbers and make direction labels consistent.
train_df["leakage_strength"] = pd.to_numeric(
    train_df["leakage_strength"]
)
train_df["leakage_direction"] = (
    train_df["leakage_direction"]
    .astype(str)
    .str.upper()
    .str.strip()
)

# Confirm the loaded size and class distributions before training.
print("TRAIN shape:", train_df.shape)
print(train_df["leakage_strength"].value_counts())
print(train_df["leakage_direction"].value_counts())


TRAIN shape: (285, 15)
leakage_strength
0    128
1    103
2     54
Name: count, dtype: int64
leakage_direction
NEUTRAL    128
CON         93
PRO         64
Name: count, dtype: int64


In [9]:
# BASELINE 1 — Load the frozen VALIDATION tab into a pandas DataFrame.
# These 44 rows are used for evaluation and threshold selection, never for updating model weights.
validation_ws = spreadsheet.worksheet("VALIDATION")

validation_df = pd.DataFrame(
    validation_ws.get_all_records()
)

print(validation_df.shape)
validation_df.head()


(61, 15)


,sentence_id,ballot_id,judge_id,actual_ballot,section,sentence,previous_sentence,next_sentence,leakage_strength,leakage_direction,needs_context,annotation_uncertain,uncertainty_reason,data_source,ballot_record_id
0,FA_R1_B2-304_S0016,FA_R1_B2-304,Hailley Yang,CON,Comments to CON,Needs more direct refutation of PRO’s economic...,The logic between economic paralysis and colla...,CON 2 : Good at picking apart CON’s examples; ...,1,PRO,,false,,additional_verified_csv_2026_09_22,additional_verified_csv_2026_09_22::FA_R1_B2-304
1,FA_R1_B2-304_S0030,FA_R1_B2-304,Hailley Yang,CON,Reason for Decision,"Con2 also claims that STEMS are wasting money,...",Some contentions are not that relevant to the ...,Con2 is also good at pointing out that Pro can...,1,PRO,,false,,additional_verified_csv_2026_09_22,additional_verified_csv_2026_09_22::FA_R1_B2-304
2,FA_R1_B2-304_S0031,FA_R1_B2-304,Hailley Yang,CON,Reason for Decision,Con2 is also good at pointing out that Pro can...,"Con2 also claims that STEMS are wasting money,...",Pro 1 can give more evidence about why exactly...,1,CON,,false,,additional_verified_csv_2026_09_22,additional_verified_csv_2026_09_22::FA_R1_B2-304
3,FA_R1_B2-304_S0033,FA_R1_B2-304,Hailley Yang,CON,Reason for Decision,Con also points out that proponents fail to gi...,Pro 1 can give more evidence about why exactly...,"Overall speaking, con made more contributions ...",1,CON,,false,,additional_verified_csv_2026_09_22,additional_verified_csv_2026_09_22::FA_R1_B2-304
4,FA_R1_B2-304_S0034,FA_R1_B2-304,Hailley Yang,CON,Reason for Decision,"Overall speaking, con made more contributions ...",Con also points out that proponents fail to gi...,,2,CON,,false,,additional_verified_csv_2026_09_22,additional_verified_csv_2026_09_22::FA_R1_B2-304


In [10]:
# BASELINE 2 — Standardize VALIDATION labels and inspect their distributions.
# Numeric strength and normalized direction values prevent type or capitalization errors in later metrics.
validation_df["leakage_strength"] = pd.to_numeric(
    validation_df["leakage_strength"]
)

validation_df["leakage_direction"] = (
    validation_df["leakage_direction"]
    .astype(str)
    .str.upper()
    .str.strip()
)

print(validation_df["leakage_strength"].value_counts())
print()
print(validation_df["leakage_direction"].value_counts())
print()
print(validation_df["section"].value_counts())


leakage_strength
0    28
1    21
2    12
Name: count, dtype: int64

leakage_direction
NEUTRAL    28
CON        19
PRO        14
Name: count, dtype: int64

section
Reason for Decision    34
Comments to CON        14
Comments to PRO        13
Name: count, dtype: int64


## Verify Split

Confirm that judge and ballot identities remain disjoint between TRAIN and VALIDATION.


In [11]:
# VERIFY SPLIT — Confirm the frozen TRAIN and VALIDATION groups do not overlap.
# TEST is deliberately not loaded, scored, sampled, or inspected anywhere in this notebook.
required_columns = {
    "sentence_id", "ballot_id", "judge_id", "sentence",
    "leakage_strength", "leakage_direction", "section"
}
assert required_columns.issubset(train_df.columns), "TRAIN is missing required columns"
assert required_columns.issubset(validation_df.columns), "VALIDATION is missing required columns"

train_judges = set(train_df["judge_id"].astype(str))
validation_judges = set(validation_df["judge_id"].astype(str))
train_ballots = set(train_df["ballot_id"].astype(str))
validation_ballots = set(validation_df["ballot_id"].astype(str))

assert train_judges.isdisjoint(validation_judges), "TRAIN/VALIDATION judge overlap"
assert train_ballots.isdisjoint(validation_ballots), "TRAIN/VALIDATION ballot overlap"

print("TRAIN/VALIDATION judge overlap:", len(train_judges & validation_judges))
print("TRAIN/VALIDATION ballot overlap:", len(train_ballots & validation_ballots))
print("TEST remains untouched and is not loaded.")


TRAIN/VALIDATION judge overlap: 0
TRAIN/VALIDATION ballot overlap: 0
TEST remains untouched and is not loaded.


## Model / Targets

Use the original paired PRO/CON target banks and exclude NEUTRAL targets from training.


In [12]:
# EXPERIMENT 1 — Define the original paired PRO and CON target banks.
# NEUTRAL targets are intentionally excluded from directional-only fine-tuning.
pro_targets = [
    "Pro wins the debate.",
    "The judge favors Pro over Con.",
    "Pro has the stronger argument than Con.",
    "Con fails against Pro's argument."
]

con_targets = [
    "Con wins the debate.",
    "The judge favors Con over Pro.",
    "Con has the stronger argument than Pro.",
    "Pro fails against Con's argument."
]


In [13]:
# EXPERIMENT 1 — Keep only real TRAIN sentences already annotated as directional leakage.
# The filter preserves every eligible annotation and excludes all NEUTRAL sentences.
directional_train_df = train_df[
    (train_df["leakage_strength"] > 0)
    & train_df["leakage_direction"].isin(["PRO", "CON"])
].copy()

directional_train_df = directional_train_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

direction_counts = directional_train_df["leakage_direction"].value_counts().reindex(["PRO", "CON"], fill_value=0)
print("Directional TRAIN distribution:")
print(direction_counts)
print("Total directional sentences:", len(directional_train_df))
assert set(directional_train_df["leakage_direction"]) <= {"PRO", "CON"}
assert (directional_train_df["leakage_strength"] > 0).all()


Directional TRAIN distribution:
leakage_direction
PRO    64
CON    93
Name: count, dtype: int64
Total directional sentences: 157


In [14]:
# EXPERIMENT 1 — Generate four paired target comparisons for every real directional sentence.
# Each anchor uses the matching target as positive and the corresponding opposite-side target as negative.
triplets = []

for _, row in directional_train_df.iterrows():
    sentence = str(row["sentence"]).strip()
    direction = row["leakage_direction"]
    if not sentence:
        continue

    for i in range(4):
        if direction == "PRO":
            positive = pro_targets[i]
            negative = con_targets[i]
        else:
            positive = con_targets[i]
            negative = pro_targets[i]

        triplets.append({
            "anchor": sentence,
            "positive": positive,
            "negative": negative,
        })

triplets_df = pd.DataFrame(triplets).sample(frac=1, random_state=42).reset_index(drop=True)
expected_triplets = 4 * directional_train_df["sentence"].astype(str).str.strip().ne("").sum()
assert len(triplets_df) == expected_triplets
print("Real directional sentences:", len(directional_train_df))
print("Generated triplets:", len(triplets_df))
display(triplets_df.head(8))


Real directional sentences: 157
Generated triplets: 628


,anchor,positive,negative
0,You did an excellent job framing the demograph...,The judge favors Con over Pro.,The judge favors Pro over Con.
1,10-day transfer vs. years of infrastructure wo...,Pro fails against Con's argument.,Con fails against Pro's argument.
2,I give my ballot to the con side.,Con has the stronger argument than Pro.,Pro has the stronger argument than Con.
3,Pro's entire case rested on welfare of women a...,The judge favors Con over Pro.,The judge favors Pro over Con.
4,Although con explains that econ benefits are n...,The judge favors Pro over Con.,The judge favors Con over Pro.
5,"However, Pro did not sufficiently prove that c...",Pro fails against Con's argument.,Con fails against Pro's argument.
6,The point about subsidies being effective unde...,The judge favors Con over Pro.,The judge favors Pro over Con.
7,Con's strongest argument is that subsidies cre...,The judge favors Con over Pro.,The judge favors Pro over Con.


In [15]:
# EXPERIMENT 1 — Fix seeds and load a fresh original BGE checkpoint.
import random
import torch
from sentence_transformers import SentenceTransformer, losses
from datasets import Dataset

EXP1_SEED = 42
random.seed(EXP1_SEED)
np.random.seed(EXP1_SEED)
torch.manual_seed(EXP1_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(EXP1_SEED)

finetune_model = SentenceTransformer("BAAI/bge-base-en-v1.5")
train_dataset = Dataset.from_pandas(
    triplets_df[["anchor", "positive", "negative"]],
    preserve_index=False
)
train_loss = losses.TripletLoss(model=finetune_model)
print("Fresh BGE loaded for Experiment 1.")
print(train_dataset)


/tmp/ipykernel_1164/505294245.py:4: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import SentenceTransformer, losses


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Fresh BGE loaded for Experiment 1.
Dataset({
    features: ['anchor', 'positive', 'negative'],
    num_rows: 628
})


## Training

Fine-tune a fresh BGE checkpoint with the original Experiment 1 TripletLoss settings.


In [16]:
# EXPERIMENT 1 — Configure the original directional-only TripletLoss run.
# The fixed settings are 2 epochs, batch size 16, learning rate 2e-5, warmup ratio 0.1, and seed 42.
from sentence_transformers import SentenceTransformerTrainer, SentenceTransformerTrainingArguments

training_args = SentenceTransformerTrainingArguments(
    output_dir="/content/bge-ballot-experiment-1",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    seed=42,
    logging_steps=5,
    save_strategy="epoch",
    report_to="none"
)

trainer = SentenceTransformerTrainer(
    model=finetune_model,
    args=training_args,
    train_dataset=train_dataset,
    loss=train_loss
)

trainer.train()

MODEL_PATH = "/content/bge-ballot-finetuned-experiment-1"
finetune_model.save(MODEL_PATH)
print("Saved Experiment 1 model to:", MODEL_PATH)


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
5,4.996066
10,4.999398
15,4.987081
20,4.985577
25,4.971707
30,4.971909
35,4.920216
40,4.926949
45,4.843970
50,4.828271


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved Experiment 1 model to: /content/bge-ballot-finetuned-experiment-1


## Validation Evaluation

Apply the historical PRO-minus-CON threshold procedure to the same frozen VALIDATION split.


In [17]:
# EXPERIMENT 1 — Reproduce the historical directional-score evaluation on frozen VALIDATION.
# A VALIDATION-selected neutral band converts the PRO-minus-CON score into PRO, CON, or NEUTRAL.
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

pro_embeddings_ft = finetune_model.encode(pro_targets, normalize_embeddings=True)
con_embeddings_ft = finetune_model.encode(con_targets, normalize_embeddings=True)
validation_sentences = validation_df["sentence"].fillna("").astype(str).tolist()
validation_embeddings_ft = finetune_model.encode(validation_sentences, normalize_embeddings=True)

validation_df["exp1_pro_score"] = (validation_embeddings_ft @ pro_embeddings_ft.T).max(axis=1)
validation_df["exp1_con_score"] = (validation_embeddings_ft @ con_embeddings_ft.T).max(axis=1)
validation_df["exp1_direction_score"] = validation_df["exp1_pro_score"] - validation_df["exp1_con_score"]

def predict_direction(score, threshold):
    if score > threshold:
        return "PRO"
    if score < -threshold:
        return "CON"
    return "NEUTRAL"

thresholds = [0.00, 0.005, 0.01, 0.015, 0.02, 0.03, 0.04, 0.05, 0.075, 0.10, 0.15]
threshold_results = []
for threshold in thresholds:
    predictions = validation_df["exp1_direction_score"].apply(
        lambda score: predict_direction(score, threshold)
    )
    threshold_results.append({
        "threshold": threshold,
        "accuracy": accuracy_score(validation_df["leakage_direction"], predictions),
        "macro_f1": f1_score(
            validation_df["leakage_direction"],
            predictions,
            average="macro",
            zero_division=0
        ),
    })

threshold_results_df = pd.DataFrame(threshold_results)
best_row = threshold_results_df.loc[threshold_results_df["macro_f1"].idxmax()]
EXP1_THRESHOLD = float(best_row["threshold"])
validation_df["exp1_prediction"] = validation_df["exp1_direction_score"].apply(
    lambda score: predict_direction(score, EXP1_THRESHOLD)
)
display(threshold_results_df.sort_values("macro_f1", ascending=False))


,threshold,accuracy,macro_f1
10,0.150,0.573770,0.561864
9,0.100,0.508197,0.493566
8,0.075,0.475410,0.457757
7,0.050,0.459016,0.434609
6,0.040,0.426230,0.386852
0,0.000,0.393443,0.333172
1,0.005,0.393443,0.333172
2,0.010,0.393443,0.333172
5,0.030,0.393443,0.333172
3,0.015,0.393443,0.333172


In [18]:
# EXPERIMENT 1 — Calculate every reported metric from the same saved prediction column.
# Historical values are sanity checks only; nothing is hard-coded into these calculations.
y_true_exp1 = validation_df["leakage_direction"]
y_pred_exp1 = validation_df["exp1_prediction"]
exp1_accuracy = accuracy_score(y_true_exp1, y_pred_exp1)
exp1_macro_f1 = f1_score(y_true_exp1, y_pred_exp1, average="macro", zero_division=0)
exp1_leakage_mask = (validation_df["leakage_strength"] > 0) & y_true_exp1.isin(["PRO", "CON"])
exp1_leakage_accuracy = accuracy_score(
    y_true_exp1[exp1_leakage_mask],
    y_pred_exp1[exp1_leakage_mask]
)

print(f"Selected VALIDATION threshold: {EXP1_THRESHOLD:.3f}")
print(f"Overall direction accuracy: {exp1_accuracy:.4f}")
print(f"Macro-F1: {exp1_macro_f1:.4f}")
print(f"Leakage-only direction accuracy: {exp1_leakage_accuracy:.4f}")
print("\nClassification report:")
print(classification_report(
    y_true_exp1,
    y_pred_exp1,
    labels=["PRO", "CON", "NEUTRAL"],
    digits=4,
    zero_division=0
))
print("Confusion matrix (rows=true, columns=predicted; PRO, CON, NEUTRAL):")
print(confusion_matrix(y_true_exp1, y_pred_exp1, labels=["PRO", "CON", "NEUTRAL"]))


Selected VALIDATION threshold: 0.150
Overall direction accuracy: 0.5738
Macro-F1: 0.5619
Leakage-only direction accuracy: 0.6970

Classification report:
              precision    recall  f1-score   support

         PRO     0.5000    0.5000    0.5000        14
         CON     0.5000    0.8421    0.6275        19
     NEUTRAL     0.8000    0.4286    0.5581        28

    accuracy                         0.5738        61
   macro avg     0.6000    0.5902    0.5619        61
weighted avg     0.6377    0.5738    0.5664        61

Confusion matrix (rows=true, columns=predicted; PRO, CON, NEUTRAL):
[[ 7  6  1]
 [ 1 16  2]
 [ 6 10 12]]


## Error Analysis

Inspect misclassified validation sentences without altering labels or the split.


In [19]:
# EXPERIMENT 1 — Display traceable VALIDATION errors for qualitative review.
# Sentence, ballot, judge, section, annotation, prediction, and similarity scores remain connected.
exp1_error_columns = [
    "sentence_id", "ballot_id", "judge_id", "section", "sentence",
    "leakage_strength", "leakage_direction", "exp1_prediction",
    "exp1_pro_score", "exp1_con_score", "exp1_direction_score"
]
exp1_errors = validation_df[
    validation_df["leakage_direction"] != validation_df["exp1_prediction"]
][exp1_error_columns].copy()
print("Total validation errors:", len(exp1_errors))
display(exp1_errors.sort_values(["leakage_strength", "ballot_id"], ascending=[False, True]))


Total validation errors: 26


,sentence_id,ballot_id,judge_id,section,sentence,leakage_strength,leakage_direction,exp1_prediction,exp1_pro_score,exp1_con_score,exp1_direction_score
48,FA_R1_B1-209_S014,FA_R1_B1-209,Anton Bezuidenhout,Reason for Decision,"They also controlled the entire debate, clearl...",2,CON,NEUTRAL,0.405384,0.334168,0.071216
17,FB_R4_3129832024_S0007,FB_R4_3129832024,Feifei Zhang,Reason for Decision,"CON’s Final Focus on cost, choice, and speed w...",2,PRO,CON,0.412773,0.936284,-0.523510
33,FA_R1_306_S006,FA_R1_306,Eric Tukei,Comments to CON,"Also, prepare short and precise questions in t...",1,PRO,CON,0.354204,0.529595,-0.175392
37,FA_R1_306_S011,FA_R1_306,Eric Tukei,Reason for Decision,The con side failed to fully rebut this.,1,PRO,CON,0.337071,0.962526,-0.625456
0,FA_R1_B2-304_S0016,FA_R1_B2-304,Hailley Yang,Comments to CON,Needs more direct refutation of PRO’s economic...,1,PRO,NEUTRAL,0.581588,0.503004,0.078584
1,FA_R1_B2-304_S0030,FA_R1_B2-304,Hailley Yang,Reason for Decision,"Con2 also claims that STEMS are wasting money,...",1,PRO,CON,0.343387,0.826316,-0.482929
7,FA_R4_3129832024_S0005,FA_R4_3129832024,Feifei Zhang,Comments to CON,Your strongest move was asking whether PRO cou...,1,CON,PRO,0.561702,0.291708,0.269994
9,FA_R4_3129832024_S0008,FA_R4_3129832024,Feifei Zhang,Reason for Decision,PRO explained well why childcare can reduce pr...,1,CON,NEUTRAL,0.540028,0.495373,0.044655
14,FB_R1_B1-209_S0031,FB_R1_B1-209,Shirley Xu,Reason for Decision,"But even con win on this clash, but because of...",1,PRO,CON,0.284290,0.917353,-0.633063
60,FB_R1_B1-209_S014,FB_R1_B1-209,Anton Bezuidenhout,Reason for Decision,The CON side could not prove how the currect s...,1,PRO,CON,0.367069,0.902505,-0.535436


## Experiment Summary

- **Data:** TRAIN rows with `leakage_strength > 0` and direction PRO or CON only; NEUTRAL training examples are intentionally excluded.
- **Training:** four paired target comparisons per real sentence, TripletLoss, 2 epochs, batch size 16, learning rate 2e-5, warmup ratio 0.1, seed 42.
- **Inference:** PRO-minus-CON similarity with a VALIDATION-selected neutral band.
- **Metrics:** overall direction accuracy, Macro-F1, leakage-only direction accuracy, classification report, and confusion matrix, all calculated from predictions.
- **TEST:** untouched and not loaded.
